In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from metrics import apk

DATA = Path("data")
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(f"CREATE VIEW tx AS SELECT * FROM '{(DATA / 'sample_transactions.parquet').as_posix()}'")
con.execute(f"CREATE VIEW customers AS SELECT * FROM '{(DATA / 'customers.parquet').as_posix()}'")

labels = con.execute(f"SELECT customer_id, bought FROM '{(DATA / 'labels_valid.parquet').as_posix()}'").df()
actual = {c: list(b) for c, b in zip(labels["customer_id"], labels["bought"])}

hist = set(con.execute("SELECT DISTINCT customer_id FROM tx WHERE w >= 2").df()["customer_id"])
cold = [c for c in actual if c not in hist]

ages = con.execute("SELECT customer_id, age FROM customers").df().set_index("customer_id")["age"]
cold_age = ages.reindex(cold)

print(f"Validation buyers: {len(actual):,} | cold-start: {len(cold):,} ({len(cold)/len(actual):.1%})")
print(f"Cold-start with missing age: {cold_age.isna().sum():,}")

Validation buyers: 14,471 | cold-start: 2,573 (17.8%)
Cold-start with missing age: 29


In [2]:
def top_items(where, k=12):
    return con.execute(f"""
        SELECT t.article_id FROM tx t JOIN customers c USING (customer_id)
        WHERE {where}
        GROUP BY t.article_id ORDER BY COUNT(DISTINCT t.customer_id) DESC, t.article_id LIMIT {k}
    """).df()["article_id"].tolist()

def age_group(a):
    if pd.isna(a): return None
    return "16-24" if a < 25 else "25-34" if a < 35 else "35-49" if a < 50 else "50+"

AGE_SQL = {"16-24": "c.age < 25", "25-34": "c.age >= 25 AND c.age < 35",
           "35-49": "c.age >= 35 AND c.age < 50", "50+": "c.age >= 50"}

pop2 = top_items("t.w BETWEEN 2 AND 3")
pop1 = top_items("t.w = 2")
pop_age = {g: top_items(f"t.w BETWEEN 2 AND 3 AND {s}") for g, s in AGE_SQL.items()}

def age_list(c):
    g = age_group(ages.get(c))
    return pop_age[g] if g else pop2

ap = {
    "Best-sellers 2 weeks": np.array([apk(actual[c], pop2) for c in cold]),
    "Best-sellers 1 week":  np.array([apk(actual[c], pop1) for c in cold]),
    "Best-sellers by age":  np.array([apk(actual[c], age_list(c)) for c in cold]),
}
for k, v in ap.items():
    print(f"{k:<22} MAP@12 = {v.mean():.5f} | customers with a hit: {(v > 0).mean():.2%}")

def paired_gain(a, b, n_boot=1000, seed=42):
    d = a - b
    rng = np.random.default_rng(seed)
    boot = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return d.mean(), lo, hi, (d > 0).sum(), (d < 0).sum()

g, lo, hi, b, w = paired_gain(ap["Best-sellers by age"], ap["Best-sellers 2 weeks"])
print(f"\nAge vs 2-week best-sellers: {g:+.5f} [{lo:+.5f}, {hi:+.5f}] | better {b} worse {w}")
print("Real gain" if lo > 0 else "Not clearly real (range includes 0)")

print("\nOverlap of age lists with the overall top 12:")
for grp, lst in pop_age.items():
    print(f"  {grp}: {len(set(lst) & set(pop2))}/12 shared")

Best-sellers 2 weeks   MAP@12 = 0.00675 | customers with a hit: 5.56%
Best-sellers 1 week    MAP@12 = 0.00689 | customers with a hit: 5.56%
Best-sellers by age    MAP@12 = 0.00778 | customers with a hit: 5.83%

Age vs 2-week best-sellers: +0.00103 [-0.00064, +0.00287] | better 96 worse 70
Not clearly real (range includes 0)

Overlap of age lists with the overall top 12:
  16-24: 8/12 shared
  25-34: 9/12 shared
  35-49: 7/12 shared
  50+: 6/12 shared


## Cold start: what new customers get

About 18% of the people I'm scoring (2,573 in the validation week) had no purchases in my 26-week window. Every method I built falls back to best-sellers for them, so I tested three versions of that fallback.

| Option | MAP@12 | Customers with a hit |
|---|---|---|
| Best-sellers, last 2 weeks | 0.00675 | 5.56% |
| Best-sellers, last 1 week | 0.00689 | 5.56% |
| Best-sellers for their age group | 0.00778 | 5.83% |

Age-based lists scored about 15% higher. I almost called that a win, but the paired bootstrap range was −0.00064 to +0.00287, which includes zero. Only 166 customers got a different result at all (96 better, 70 worse), which is too few to be sure. That fits the EDA, where age had only a weak link with what people buy (Cramér's V of 0.064).

So for now I'd keep plain 2-week best-sellers for new customers. Age-based lists are cheap to build and point in the right direction, so they're a good candidate for a live A/B test rather than something to switch on from one offline week. The 29 new customers with no age would get the normal best-sellers either way.

One thing that looks better: the LightGBM ranker scored 0.00753 on customers without recent history in Step 10, because it picks from 30 popular items using last week's trends. That group was defined slightly differently (no purchases in 8 weeks rather than 24), so the numbers aren't directly comparable.